In [10]:
!git clone https://github.com/Eleftheria111/LAHIS.git
%cd LAHIS
!ls
!find . -maxdepth 2 -type f | head -100

Cloning into 'LAHIS'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 90 (delta 17), reused 73 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 1.29 MiB | 32.94 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/LAHIS/LAHIS
data  LAHIS_pipeline.ipynb  LAHIS_pipeline_olmo2.ipynb	README.md  results  src
./LAHIS_pipeline.ipynb
./src/intervention_demo.py
./src/model_handler.py
./src/attn_matrix_ted.py
./src/heatmap_viz.py
./src/ted_loader.py
./src/multilingual_exps.py
./src/lan_head_mask.py
./src/run_pipeline.py
./src/language_heads.py
./src/specificity_eval.py
./src/attn_matrix.py
./.gitignore
./README.md
./LAHIS_pipeline_olmo2.ipynb
./.git/index
./.git/packed-refs
./.git/description
./.git/config
./.git/HEAD


In [11]:
!pip install -U torch transformers accelerate peft sentencepiece safetensors matplotlib python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 128.7 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")



True
Tesla T4


In [2]:
import os

if not os.path.isdir('/content/LAHIS'):
    !git clone https://github.com/Eleftheria111/LAHIS.git /content/LAHIS

%cd /content/LAHIS
!ls


/content/LAHIS
data   LAHIS_pipeline.ipynb	   README.md  src
LAHIS  LAHIS_pipeline_olmo2.ipynb  results


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# LAHIS — Language Attention Head Importance Scores
### OLMo-2-7B × TED Multilingual Data

This notebook applies the LAHIS method to **OLMo-2-7B**.

**Steps:**
1. Install dependencies
2. Imports and setup
3. Config — language list aligned with Jonny's finetuning languages
4. Prepare TED data
5. Load OLMo-2-7B with custom LAHIS head-mask patch
6. Compute importance matrices (base model)
7. Heatmaps
8. Head distribution by layer
9. Language-specific vs language-general heads
10. Specificity validation — dark diagonal
11. Base vs finetuned comparison (run after Jonny uploads to HuggingFace)

In [4]:
## 1 · Install dependencies
import subprocess, sys
pkgs = ["datasets", "transformers", "torch", "tqdm", "matplotlib", "huggingface_hub", "python-dotenv"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'datasets', 'transformers', 'torch', 'tqdm', 'matplotlib', 'huggingface_hub', 'python-dotenv'], returncode=0)

In [5]:
## 2 · Imports and setup
import os, sys, json, gc, torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from IPython.display import display, Image
from tqdm.notebook import tqdm as tqdm_nb

# Navigate to LAHIS/src/ regardless of where the kernel started
cwd = os.path.abspath('')
if os.path.basename(cwd) == 'src':
    SRC_DIR = cwd
elif os.path.isdir(os.path.join(cwd, 'src')):
    SRC_DIR = os.path.join(cwd, 'src')
else:
    raise FileNotFoundError(f"Cannot find src/ from {cwd} — open the notebook from LAHIS/")

os.chdir(SRC_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('Working directory:', os.getcwd())
print('Python         :', sys.version.split()[0])
print('PyTorch        :', torch.__version__)
print('MPS available  :', torch.backends.mps.is_available())
print('CUDA available :', torch.cuda.is_available())

Working directory: /content/LAHIS/src
Python         : 3.12.12
PyTorch        : 2.10.0+cu128
MPS available  : False
CUDA available : True


In [6]:
## 3 · Config

# ══════════════════════════════════════════════════════════════════════════════
# QUICK_TEST — set to True for the test, False for real results
#
#   True  → DATA_NUM=10, MAX_LENGTH=64, SPEC_DATA_NUM=10  I used this to make sure that everything run properly
#   False → DATA_NUM=1000, MAX_LENGTH=512, SPEC_DATA_NUM=500  (Jonny's GPU)
# ══════════════════════════════════════════════════════════════════════════════
QUICK_TEST = False   # flip to False before running

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'
print('Using device:', DEVICE)

LANGUAGES = ["en", "de", "id", "pt", "ar", "bn", "sw", "es", "ru", "fr", "ja", "zh"]

if QUICK_TEST:
    DATA_NUM      = 10
    MAX_LENGTH    = 64
    SPEC_DATA_NUM = 10
else:
    DATA_NUM      = 1000
    MAX_LENGTH    = 512
    SPEC_DATA_NUM = 500

TED_JSONL = '../../TED2025/multi_way.jsonl'
DATA_DIR  = '../data/ted'

if QUICK_TEST:
    RESULTS_DIR = '../results/olmo2_quicktest'
    HEATMAP_DIR = '../results/olmo2_quicktest/heatmaps'
else:
    RESULTS_DIR = '../results/olmo2'
    HEATMAP_DIR = '../results/olmo2/heatmaps'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)

TOP_P = 0.02

print(f'QUICK_TEST    : {QUICK_TEST}')
print(f'DATA_NUM      : {DATA_NUM}')
print(f'MAX_LENGTH    : {MAX_LENGTH}')
print(f'SPEC_DATA_NUM : {SPEC_DATA_NUM}')
print(f'Results dir   : {RESULTS_DIR}')

Using device: cuda
QUICK_TEST    : False
DATA_NUM      : 1000
MAX_LENGTH    : 512
SPEC_DATA_NUM : 500
Results dir   : ../results/olmo2


## 4 · Prepare TED Data
Converts the raw `multi_way.jsonl` into one JSON file per language.  
Each file is a list of `{"text": "..."}` chunks (5 sentences joined together).

**Notes on Jonny's language list:**
- `en, de, ar, es, ru, fr, ja, zh` → already built if you ran the Llama-2 notebook
- `id` (Indonesian), `pt` (Portuguese) → moderate resource, should have good TED coverage
- `bn` (Bengali), `sw` (Swahili) → low-resource; if < 1000 chunks exist the importance matrix will just use however many are available

**Skip this cell if all JSON files already exist.**

In [8]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [11]:
!find /content/drive/MyDrive -name "multi_way.jsonl"
TED_JSONL = "/content/drive/MyDrive/multi_way.jsonl"
import os
print(os.path.exists(TED_JSONL), TED_JSONL)


/content/drive/MyDrive/multi_way.jsonl
True /content/drive/MyDrive/multi_way.jsonl


In [12]:
import ted_loader as TED

missing = [l for l in LANGUAGES if not os.path.exists(os.path.join(DATA_DIR, f'ted_{l}.json'))]

if not missing:
    print('All TED data files already exist — skipping.')
else:
    print(f'Building data for: {missing}')
    records = TED.load_ted_jsonl(TED_JSONL)
    print(f'Loaded {len(records):,} TED records')
    streams = TED.build_monolingual_streams(records, missing, join_n_sentences=5)
    TED.save_monolingual_json(streams, DATA_DIR)
    print('Done.')

# Show summary — mark missing ones
print('\nData files:')
AVAILABLE_LANGS = []
for lang in LANGUAGES:
    path = os.path.join(DATA_DIR, f'ted_{lang}.json')
    if os.path.exists(path):
        size = os.path.getsize(path) // (1024 * 1024)
        print(f'  {lang:6s}  {size:4d} MB')
        AVAILABLE_LANGS.append(lang)
    else:
        print(f'  {lang:6s}  MISSING — excluded from analysis')

print(f'\nWill run LAHIS on: {AVAILABLE_LANGS}')

Building data for: ['en', 'de', 'id', 'pt', 'ar', 'bn', 'sw', 'es', 'ru', 'fr', 'ja', 'zh']
Loaded 2,218,563 TED records
  [ar      ] 235,033 chunks  ->  ../data/ted/ted_ar.json
  [bn      ]  5,180 chunks  ->  ../data/ted/ted_bn.json
  [de      ] 116,063 chunks  ->  ../data/ted/ted_de.json
  [en      ] 319,515 chunks  ->  ../data/ted/ted_en.json
  [es      ] 250,755 chunks  ->  ../data/ted/ted_es.json
  [fr      ] 207,182 chunks  ->  ../data/ted/ted_fr.json
  [id      ] 90,731 chunks  ->  ../data/ted/ted_id.json
  [ja      ] 63,095 chunks  ->  ../data/ted/ted_ja.json
  [pt      ] 279,528 chunks  ->  ../data/ted/ted_pt.json
  [ru      ] 176,069 chunks  ->  ../data/ted/ted_ru.json
  [sw      ]  3,861 chunks  ->  ../data/ted/ted_sw.json
  [zh      ] 39,241 chunks  ->  ../data/ted/ted_zh.json
Done.

Data files:
  en        78 MB
  de        29 MB
  id        22 MB
  pt        70 MB
  ar        86 MB
  bn         3 MB
  sw         0 MB
  es        63 MB
  ru        71 MB
  fr        54 MB
 

## 5 · Load OLMo-2-7B
Downloads from HuggingFace on first run (~14 GB, open weights — **no token needed**).  
The custom OLMo-2 LAHIS patch is applied automatically. It is identical to the Llama-2 patch but adds `q_norm` / `k_norm` (RMSNorm applied per-head to queries and keys — OLMo-2 specific).

In [ ]:
import importlib
import model_handler
importlib.reload(model_handler)

model, tokenizer = model_handler.load_model(
    'olmo2',
    device=DEVICE,
    half_precision=True,
    local=False,
)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers   # 32
NUM_HEADS  = model.config.num_attention_heads  # 32
print(f'\nModel : OLMo-2-7B')
print(f'Layers: {NUM_LAYERS}  ×  Heads: {NUM_HEADS}  =  {NUM_LAYERS * NUM_HEADS} total heads')
print('LAHIS patch applied:', hasattr(model.model.layers[0].self_attn, '_lahis_mask'))

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Load model in torch.bfloat16


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

## 6 · Compute LAHIS Importance Matrices (Base Model)
For each language, run a forward+backward pass over TED text samples.  
The gradient of the soft head mask gives the importance of each head for that language.

**Output:** `../results/olmo2/olmo2_{lang}.pth` — a `[32 × 32]` tensor per language.  
**Cached:** if the `.pth` file already exists the matrix is loaded instead of recomputed.

In [ ]:
from attn_matrix_ted import get_attn_head_matrix_ted

importance_matrices = {}

for lang in AVAILABLE_LANGS:
    out_path = os.path.join(RESULTS_DIR, f'olmo2_{lang}.pth')

    if os.path.exists(out_path):
        print(f'[{lang}] Loading cached matrix')
        importance_matrices[lang] = torch.load(out_path, map_location='cpu').float()
        m = importance_matrices[lang]
        print(f'      Score range: [{m.min():.2f}, {m.max():.2f}]')
    else:
        print(f'\n[{lang}] Computing importance matrix ({DATA_NUM} samples)...')
        matrix = get_attn_head_matrix_ted(
            model, tokenizer,
            lan=lang,
            model_name='olmo2',
            data_dir=DATA_DIR,
            data_num=DATA_NUM,
            max_length=MAX_LENGTH,
        )
        importance_matrices[lang] = matrix.float()
        torch.save(matrix, out_path)
        print(f'      Saved → {out_path}')

print(f'\nMatrices ready for: {list(importance_matrices.keys())}')

## 7 · Heatmaps
Each cell in the grid is one attention head. Brighter = more important for that language.  
Compare with the Llama-2 heatmaps to see whether OLMo-2 (more English-dominant) shows weaker or differently located language-specific patterns.

In [ ]:
langs = list(importance_matrices.keys())
n     = len(langs)
ncols = min(n, 5)
nrows = (n + ncols - 1) // ncols

all_vals = torch.cat([importance_matrices[l].view(-1) for l in langs])
vmin, vmax = all_vals.min().item(), all_vals.max().item()

fig = plt.figure(figsize=(ncols * 4, nrows * 3.2))
gs  = gridspec.GridSpec(nrows, ncols + 1,
                        width_ratios=[1]*ncols + [0.05],
                        hspace=0.5, wspace=0.3)
last_im = None

for i, lang in enumerate(langs):
    row, col = divmod(i, ncols)
    ax  = fig.add_subplot(gs[row, col])
    mat = importance_matrices[lang].numpy()
    last_im = ax.imshow(mat, aspect='auto', cmap='YlOrRd',
                        vmin=vmin, vmax=vmax, origin='upper')
    ax.set_title(lang.upper(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Head', fontsize=8)
    ax.set_ylabel('Layer', fontsize=8)
    ax.set_xticks(range(0, NUM_HEADS, 4))
    ax.set_yticks(range(0, NUM_LAYERS, 4))
    ax.tick_params(labelsize=7)

for j in range(len(langs), nrows * ncols):
    row, col = divmod(j, ncols)
    fig.add_subplot(gs[row, col]).axis('off')

cbar_ax = fig.add_subplot(gs[:, ncols])
fig.colorbar(last_im, cax=cbar_ax, label='LAHIS importance')
fig.suptitle(f'LAHIS Head Importance — OLMo-2-7B (base) + TED\n({NUM_LAYERS} layers × {NUM_HEADS} heads)',
             fontsize=14, y=1.02)

out_path = os.path.join(HEATMAP_DIR, 'all_languages_base.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out_path}')
display(Image(out_path))

## 8 · Head Distribution by Layer
For each language, where in the network (which layers) do the top-2% most important heads cluster?  
For Llama-2 we saw: European languages → layers 0–2, Non-Latin scripts → around layer 16.

In [ ]:
n_top = max(1, int(NUM_LAYERS * NUM_HEADS * TOP_P))

fig, ax = plt.subplots(figsize=(11, 5))

for lang in langs:
    matrix = importance_matrices[lang]
    _, topk_idx = torch.topk(matrix.view(-1), k=n_top)

    layer_counts = torch.zeros(NUM_LAYERS, dtype=torch.int)
    for idx in topk_idx.tolist():
        l, _ = divmod(idx, NUM_HEADS)
        layer_counts[l] += 1

    pct = (layer_counts.float() / layer_counts.sum() * 100).numpy()
    ax.plot(range(NUM_LAYERS), pct, '-o', markersize=4, label=lang.upper())

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel(f'% of top-{int(TOP_P*100)}% heads in this layer', fontsize=11)
ax.set_title('OLMo-2-7B (base): Where do language-important heads cluster?', fontsize=13)
ax.legend(ncol=6, fontsize=9)
ax.grid(alpha=0.3)

out_path = os.path.join(HEATMAP_DIR, 'head_distribution_base.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out_path}')
display(Image(out_path))

## 9 · Language-Specific and Language-General Heads
- **Language-general heads:** top-2% for the majority of languages — handle universal grammar/syntax.  
- **Language-specific heads:** top-2% for one language but NOT for most others — the "language gate" we care about.

Saved to `../results/olmo2/head_indices_base.json` and `repeated_indices_base.json`.

In [ ]:
from collections import Counter

Q       = 1.0 - TOP_P
MIN_REP = max(2, len(langs) - 1)   # appear in N-1 languages = general

sorted_head_list = []
for lang in langs:
    flat      = importance_matrices[lang].view(-1).float()
    threshold = torch.quantile(flat, q=Q)
    topk_idx  = (flat > threshold).nonzero(as_tuple=False).squeeze()
    if topk_idx.dim() == 0:
        topk_idx = topk_idx.unsqueeze(0)
    vals  = flat[topk_idx]
    order = torch.argsort(vals, descending=True)
    sorted_head_list.append(topk_idx[order])

all_indices  = torch.cat(sorted_head_list).tolist()
index_counts = Counter(all_indices)
general_heads = [
    idx for idx, cnt in sorted(index_counts.items(), key=lambda x: -x[1])
    if cnt >= MIN_REP
]

general_set    = set(general_heads)
specific_heads = {}
for lang, topk in zip(langs, sorted_head_list):
    specific_heads[lang] = [idx for idx in topk.tolist() if idx not in general_set]

with open(os.path.join(RESULTS_DIR, 'head_indices_base.json'), 'w') as f:
    json.dump(specific_heads, f, indent=2)
with open(os.path.join(RESULTS_DIR, 'repeated_indices_base.json'), 'w') as f:
    json.dump(general_heads, f, indent=2)

print(f'Language-general heads (in top-2% for {MIN_REP}+ languages): {len(general_heads)}')
print(f'  First 10 (layer, head): {[(idx // NUM_HEADS, idx % NUM_HEADS) for idx in general_heads[:10]]}')

print(f'\nLanguage-specific heads (top-5 per language):')
for lang in langs:
    top5 = [(idx // NUM_HEADS, idx % NUM_HEADS) for idx in specific_heads[lang][:5]]
    print(f'  {lang.upper():6s}: {len(specific_heads[lang]):3d} heads   top-5: {top5}')

## 10 · Specificity Validation — Dark Diagonal
Disable the top-2% heads for each language and measure perplexity across all languages.  
**Expected result:** disabling language X heads hurts language X the most → dark diagonal.

This cell takes a while (it runs the model ~N² times). Use `SPEC_DATA_NUM = 500` for publishable results.

In [ ]:
import datasets as hf_datasets

if QUICK_TEST:
    print("QUICK_TEST=True — skipping.")
else:
    def ppl_with_mask(test_lang, head_mask):
        data_file = os.path.join(DATA_DIR, f'ted_{test_lang}.json')
        ds = hf_datasets.load_dataset('json', data_files=data_file, split='train')
        ds = ds.shuffle(seed=37).select(range(min(SPEC_DATA_NUM, len(ds))))
        hm = head_mask.to(model.device).to(model.dtype)
        model_handler.set_lahis_head_mask(model, hm)
        total, count = 0.0, 0
        for item in ds:
            ids = tokenizer(item['text'], return_tensors='pt',
                            truncation=True, max_length=MAX_LENGTH).input_ids.to(model.device)
            with torch.no_grad():
                out = model(ids, labels=ids)
                total += torch.exp(out.loss).item()
                count += 1
        model_handler.clear_lahis_head_mask(model)
        return round(total / max(count, 1), 2)

    spec_path = os.path.join(RESULTS_DIR, 'specificity_base.json')
    if os.path.exists(spec_path):
        print(f'Loading cached results from {spec_path}')
        with open(spec_path) as f:
            spec_results = json.load(f)
    else:
        print('Computing baseline perplexity (no masking)...')
        ori_mask = torch.ones(NUM_LAYERS, NUM_HEADS)
        baseline = {l: ppl_with_mask(l, ori_mask) for l in tqdm_nb(langs, desc='baseline')}
        print('Baseline PPL:', baseline)
        print('\nComputing masked perplexity...')
        masked = {}
        for head_lang in tqdm_nb(langs, desc='head_lang'):
            matrix  = importance_matrices[head_lang]
            n_top_s = max(1, int(NUM_LAYERS * NUM_HEADS * TOP_P))
            _, topk_idx = torch.topk(matrix.view(-1), k=n_top_s)
            hm = torch.ones(NUM_LAYERS, NUM_HEADS)
            hm.view(-1)[topk_idx] = 0.0
            masked[head_lang] = {tl: ppl_with_mask(tl, hm) for tl in langs}
        delta = {hl: {tl: round(masked[hl][tl] - baseline[tl], 2) for tl in langs} for hl in langs}
        spec_results = {'baseline': baseline, 'masked': masked, 'delta': delta}
        with open(spec_path, 'w') as f:
            json.dump(spec_results, f, indent=2)
        print(f'Saved → {spec_path}')

    eval_langs = list(spec_results['delta'].keys())
    n_l = len(eval_langs)
    mat = np.array([[spec_results['delta'][hl][tl] for tl in eval_langs] for hl in eval_langs])
    fig, ax = plt.subplots(figsize=(max(6, n_l), max(5, n_l)))
    im = ax.imshow(mat, cmap='Reds', aspect='auto')
    ax.set_xticks(range(n_l)); ax.set_xticklabels([l.upper() for l in eval_langs], fontsize=9)
    ax.set_yticks(range(n_l)); ax.set_yticklabels([l.upper() for l in eval_langs], fontsize=9)
    ax.set_xlabel('Language tested', fontsize=11)
    ax.set_ylabel('Language whose heads are disabled', fontsize=11)
    ax.set_title('Specificity: PPL increase when top-2% heads disabled', fontsize=12)
    for i in range(n_l):
        for j in range(n_l):
            colour = 'white' if mat[i, j] > mat.max() * 0.6 else 'black'
            ax.text(j, i, f'{mat[i,j]:.1f}', ha='center', va='center', fontsize=8, color=colour)
    fig.colorbar(im, ax=ax, label='PPL increase vs baseline')
    out_path = os.path.join(HEATMAP_DIR, 'dark_diagonal_base.png')
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved → {out_path}')
    display(Image(out_path))


---
## 11 · Base vs Finetuned Comparison

1. Set `FINETUNED_HF_PATH` to his model path (e.g. `"hf:jonny-username/olmo2-finetuned-parallel"`)
2. Run this cell

The delta heatmap (`finetuned − base`) shows which heads were strengthened by LoRA.  
If language-specific heads for the finetuned language X become brighter, that is mechanistic evidence for our gating hypothesis.

In [ ]:
# PUT YOUR MODEL PATH HERE once it's on HuggingFace
FINETUNED_HF_PATH = "hf:GBBurgess/olmo2-lora-ft"

FT_RESULTS_DIR = '../results/olmo2_finetuned'
FT_HEATMAP_DIR = '../results/olmo2_finetuned/heatmaps'

if FINETUNED_HF_PATH is None:
    print("Set FINETUNED_HF_PATH above and re-run this cell.")
else:
    os.makedirs(FT_RESULTS_DIR, exist_ok=True)
    os.makedirs(FT_HEATMAP_DIR, exist_ok=True)

    # 1. free base model memory then load finetuned
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()

    ft_model, ft_tokenizer = model_handler.load_model(
        FINETUNED_HF_PATH, device=DEVICE, half_precision=True, local=False
    )
    ft_model.eval()
    print(f'Loaded: {FINETUNED_HF_PATH}')

    # 2. compute importance matrices
    ft_matrices = {}
    for lang in langs:
        out_path = os.path.join(FT_RESULTS_DIR, f'olmo2_ft_{lang}.pth')
        if os.path.exists(out_path):
            print(f'[{lang}] loading cached')
            ft_matrices[lang] = torch.load(out_path, map_location='cpu').float()
        else:
            print(f'[{lang}] computing...')
            matrix = get_attn_head_matrix_ted(
                ft_model, ft_tokenizer, lan=lang, model_name='olmo2_ft',
                data_dir=DATA_DIR, data_num=DATA_NUM, max_length=MAX_LENGTH,
            )
            ft_matrices[lang] = matrix.float()
            torch.save(matrix, out_path)
    print('Matrices done.')

    # 3. heatmaps (finetuned)
    ft_langs = list(ft_matrices.keys())
    n = len(ft_langs); ncols = min(n, 5); nrows = (n + ncols - 1) // ncols
    all_vals = torch.cat([ft_matrices[l].view(-1) for l in ft_langs])
    vmin, vmax = all_vals.min().item(), all_vals.max().item()
    fig = plt.figure(figsize=(ncols * 4, nrows * 3.2))
    gs  = gridspec.GridSpec(nrows, ncols + 1, width_ratios=[1]*ncols + [0.05], hspace=0.5, wspace=0.3)
    last_im = None
    for i, lang in enumerate(ft_langs):
        row, col = divmod(i, ncols)
        ax = fig.add_subplot(gs[row, col])
        last_im = ax.imshow(ft_matrices[lang].numpy(), aspect='auto', cmap='YlOrRd',
                            vmin=vmin, vmax=vmax, origin='upper')
        ax.set_title(lang.upper(), fontsize=12, fontweight='bold')
        ax.set_xlabel('Head', fontsize=8); ax.set_ylabel('Layer', fontsize=8)
        ax.tick_params(labelsize=7)
    fig.colorbar(last_im, cax=fig.add_subplot(gs[:, ncols]), label='LAHIS importance')
    fig.suptitle('LAHIS Head Importance — OLMo-2-7B (finetuned)', fontsize=14, y=1.02)
    p = os.path.join(FT_HEATMAP_DIR, 'all_languages_finetuned.png')
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'Saved → {p}'); display(Image(p))

    # 4. head distribution by layer (finetuned)
    fig, ax = plt.subplots(figsize=(11, 5))
    for lang in ft_langs:
        _, topk_idx = torch.topk(ft_matrices[lang].view(-1), k=max(1, int(NUM_LAYERS * NUM_HEADS * TOP_P)))
        layer_counts = torch.zeros(NUM_LAYERS, dtype=torch.int)
        for idx in topk_idx.tolist():
            l, _ = divmod(idx, NUM_HEADS)
            layer_counts[l] += 1
        pct = (layer_counts.float() / layer_counts.sum() * 100).numpy()
        ax.plot(range(NUM_LAYERS), pct, '-o', markersize=4, label=lang.upper())
    ax.set_xlabel('Layer', fontsize=12)
    ax.set_ylabel(f'% of top-{int(TOP_P*100)}% heads', fontsize=11)
    ax.set_title('OLMo-2-7B (finetuned): Where do language-important heads cluster?', fontsize=13)
    ax.legend(ncol=6, fontsize=9); ax.grid(alpha=0.3)
    p = os.path.join(FT_HEATMAP_DIR, 'head_distribution_finetuned.png')
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'Saved → {p}'); display(Image(p))

    # 5. head selection (finetuned)
    from collections import Counter
    ft_sorted_head_list = []
    for lang in ft_langs:
        flat = ft_matrices[lang].view(-1).float()
        threshold = torch.quantile(flat, q=1.0 - TOP_P)
        topk_idx = (flat > threshold).nonzero(as_tuple=False).squeeze()
        if topk_idx.dim() == 0: topk_idx = topk_idx.unsqueeze(0)
        ft_sorted_head_list.append(topk_idx[torch.argsort(flat[topk_idx], descending=True)])

    ft_index_counts = Counter(torch.cat(ft_sorted_head_list).tolist())
    ft_general_heads = [idx for idx, cnt in sorted(ft_index_counts.items(), key=lambda x: -x[1])
                        if cnt >= max(2, len(ft_langs) - 1)]
    ft_general_set = set(ft_general_heads)
    ft_specific_heads = {lang: [idx for idx in topk.tolist() if idx not in ft_general_set]
                         for lang, topk in zip(ft_langs, ft_sorted_head_list)}

    with open(os.path.join(FT_RESULTS_DIR, 'head_indices_finetuned.json'), 'w') as f:
        json.dump(ft_specific_heads, f, indent=2)
    with open(os.path.join(FT_RESULTS_DIR, 'repeated_indices_finetuned.json'), 'w') as f:
        json.dump(ft_general_heads, f, indent=2)

    print(f'\nGeneral heads (finetuned): {len(ft_general_heads)}  vs base: {len(general_heads)}')
    print('Specific heads top-5 per language:')
    for lang in ft_langs:
        top5 = [(idx // NUM_HEADS, idx % NUM_HEADS) for idx in ft_specific_heads[lang][:5]]
        print(f'  {lang.upper():6s}: {len(ft_specific_heads[lang]):3d} heads   top-5: {top5}')

    # 6. delta heatmap: finetuned − base (the main paper figure)
    delta_langs = [l for l in langs if l in ft_matrices and l in importance_matrices]
    n = len(delta_langs); ncols = min(n, 5); nrows = (n + ncols - 1) // ncols
    fig = plt.figure(figsize=(ncols * 4, nrows * 3.2))
    gs  = gridspec.GridSpec(nrows, ncols + 1, width_ratios=[1]*ncols + [0.05], hspace=0.5, wspace=0.3)
    last_im = None
    for i, lang in enumerate(delta_langs):
        delta_mat = (ft_matrices[lang] - importance_matrices[lang]).numpy()
        row, col  = divmod(i, ncols)
        ax = fig.add_subplot(gs[row, col])
        vabs = max(abs(delta_mat.min()), abs(delta_mat.max()))
        last_im = ax.imshow(delta_mat, aspect='auto', cmap='RdBu_r',
                            vmin=-vabs, vmax=vabs, origin='upper')
        ax.set_title(lang.upper(), fontsize=12, fontweight='bold')
        ax.set_xlabel('Head', fontsize=8); ax.set_ylabel('Layer', fontsize=8)
        ax.tick_params(labelsize=7)
    fig.colorbar(last_im, cax=fig.add_subplot(gs[:, ncols]), label='Δ importance (finetuned − base)')
    fig.suptitle('OLMo-2-7B: Finetuned − Base\n(red = head strengthened by LoRA)', fontsize=13, y=1.02)
    p = os.path.join(FT_HEATMAP_DIR, 'delta_ft_vs_base.png')
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close(fig)
    print(f'Saved → {p}'); display(Image(p))